In [2]:
import matplotlib.pyplot as plt
import geopandas
import pandas as pd
import sqlite3
from sklearn.preprocessing import StandardScaler, MinMaxScaler
geopandas.options.io_engine = "pyogrio"

conn = sqlite3.connect('db.sqlite3')
cursor = conn.cursor()

data = conn.execute("""
with attendances_month as (
select
    sum(data.value) as value,
    data.commune_id as commune_id,
    data.report_id as report_id,
    data.year || '-' || (
        case data.cohort
            when 'Enero' then '01'
            when 'Febrero' then '02'
            when 'Marzo' then '03'
            when 'Abril' then '04'
            when 'Mayo' then '05'
            when 'Junio' then '06'
            when 'Julio' then '07'
            when 'Agosto' then '08'
            when 'Septiembre' then '09'
            when 'Octubre' then '10'
            when 'Noviembre' then '11'
            when 'Diciembre' then '12'
        end
    ) || '-01' as date
from data
where cohort in ('Enero', 'Febrero', 'Marzo', 'Abril', 'Mayo', 'Junio', 'Julio', 'Agosto', 'Septiembre', 'Octubre', 'Noviembre', 'Diciembre')
  and data.year > 2020
group by date, commune_id, report_id
),
aqi_month as (
select
    commune_id,
    date(datetime, 'unixepoch', 'start of month') as date, -- collapse entire month to first day of month
    avg(case when contaminant = 'CO' then concentration end)     as CO,
    avg(case when contaminant = 'MP10' then concentration end)   as MP10,
    avg(case when contaminant = 'MP2.5' then concentration end)  as MP25,
    avg(case when contaminant = 'NO' then concentration end)     as NO,
    avg(case when contaminant = 'NO2' then concentration end)    as NO2,
    avg(case when contaminant = 'NOx' then concentration end)    as NOx,
    avg(case when contaminant = 'O3' then concentration end)     as O3,
    avg(case when contaminant = 'SO2' then concentration end)    as SO2
from contaminant
group by date, commune_id
)
select 
    attendances_month.date as date,
    commune.name as commune,
    attendances_month.value * 10000 / commune.population as attendances_10k,
    aqi_month.CO as CO,
    aqi_month.MP10 as MP10,
    aqi_month.MP25 as MP25,
    aqi_month.NO as NO,
    aqi_month.NO2 as NO2,
    aqi_month.NOx as NOx,
    aqi_month.O3 as O3,
    aqi_month.SO2 as SO2
from attendances_month
join aqi_month on aqi_month.date = attendances_month.date and aqi_month.commune_id = attendances_month.commune_id
join commune on commune.id = attendances_month.commune_id
join report on report.id = attendances_month.report_id
  and report.description = 'Ingresos Programa de Salud Mental por mes y año'
""").fetchall()
# and attendances_month.date not in ('2020-01-01', '2020-02-01', '2020-03-01', '2020-04-01', '2020-05-01', '2020-06-01', '2020-07-01', '2020-08-01', '2020-09-01', '2020-10-01', '2020-11-01', '2020-12-01') -- remove 2020 because of covid

data = pd.DataFrame(data, columns=['date', 'commune', 'Ingresos Programa de Salud Mental por mes y año', 'CO', 'MP10', 'MP25', 'NO', 'NO2', 'NOx', 'O3', 'SO2'])

data


,date,commune,Ingresos Programa de Salud Mental por mes y año,CO,MP10,MP25,NO,NO2,NOx,O3,SO2
0,2021-01-01,LA CISTERNA,16,0.526000,100.903226,27.586207,37.044306,23.140334,44.154964,NaN,2.254706
1,2021-01-01,PENALOLEN,19,0.832667,79.310345,22.448276,29.721776,26.376333,51.739570,51.225806,1.008000
2,2021-01-01,SANTIAGO,6,0.808000,87.000000,23.612903,40.887408,27.035161,65.935815,NaN,NaN
3,2021-01-01,PUENTE ALTO,10,0.589667,97.133333,29.000000,NaN,NaN,NaN,50.200000,1.286333
4,2021-01-01,CERRO NAVIA,16,0.563500,81.887097,22.677419,24.598882,19.411639,42.767826,42.967742,NaN
...,...,...,...,...,...,...,...,...,...,...,...
379,2024-08-01,CERRILLOS,14,NaN,153.961538,68.576923,NaN,NaN,NaN,NaN,NaN
380,2024-08-01,CERRO NAVIA,28,NaN,163.521739,69.666667,NaN,NaN,NaN,NaN,NaN
381,2024-08-01,LO BARNECHEA,17,NaN,89.714286,45.785714,NaN,NaN,NaN,NaN,NaN
382,2024-08-01,QUILICURA,23,NaN,125.434783,53.040000,NaN,NaN,NaN,NaN,NaN


In [10]:
# check correlation
corrs = []
for commune in data['commune'].unique():
    commune_data = data[data['commune'] == commune]
    corr = commune_data.drop(columns=['date', 'commune']).corr().iloc[0]
    # get first row
    corrs.append(corr)

corrs_df = pd.DataFrame(corrs, index=data['commune'].unique())
corrs_df.mean()

Ingresos Programa de Salud Mental por mes y año    1.000000
CO                                                -0.001079
MP10                                               0.013134
MP25                                              -0.016779
NO                                                 0.023387
NO2                                               -0.027419
NOx                                                0.017604
O3                                                -0.128139
SO2                                                0.502481
dtype: float64